# EUR/USD: del Blueprint al notebook

Reproduce las siete ramas del blueprint dentro del motor PyTorch del proyecto. El original usa Keras. Las métricas de ajuste completo son in-sample; no demuestran capacidad de pronóstico.

Ejecuta con el Python de `quant-blueprint-studio/.venv` y los requisitos del proyecto instalados. No se ejecuta el notebook adjunto original.


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, SVG

roots = [Path.cwd(), Path.cwd()/"quant-blueprint-studio", Path.cwd().parent, Path.cwd().parent/"quant-blueprint-studio"]
root = next((p for p in roots if (p/"backend/engine/dag_runner.py").is_file()), None)
if root is None:
    raise RuntimeError("Abre este notebook desde el proyecto quant-blueprint-studio o su carpeta examples.")
sys.path.insert(0, str(root))
from backend.engine.dag_runner import DAGRunner


## 1. Elige el diseño

`quick`: 100 épocas; `reference`: 1500/2000/2500; `temporal`: TR/VA/TE. La fuente predeterminada descarga EURUSD=X. Selecciona `offline=True` para una demostración sintética.


In [ ]:
preset = "quick"  # quick / reference / temporal
offline = False
if preset not in {"quick", "reference", "temporal"}:
    raise ValueError("Preset desconocido")
entry = json.loads((root/f"public/blueprints/notebook-curve-{preset}.json").read_text())["entries"][0]
graph = entry["graph"]
if offline:
    for node in graph["nodes"]:
        if node["data"]["kind"] == "source":
            node["data"]["params"]["source"] = "synthetic"
display(pd.DataFrame([{ "node":n["id"], "kind":n["data"]["kind"], "parameters":n["data"]["params"] } for n in graph["nodes"]]))


## 2. Ejecuta el mismo DAG

Puedes modificar los parámetros de las ramas en `graph` antes de ejecutar. El benchmark incluye errores alineados, activación, optimizador, parámetros, épocas y tiempo.


In [ ]:
runner = DAGRunner(graph)
result = runner.execute()
benchmark = pd.DataFrame([row for study in result["studies"] for row in study["table"]])
display(benchmark)


## 3. Inspecciona datos y curvas

Los precios mensuales sin centrar se mantienen en el nodo monthly. La siguiente figura dibuja las curvas ajustadas por cada rama.


In [ ]:
display(runner.outputs["monthly"].head())
from html import escape
colors=["#16a085","#8e44ad","#d35400","#2980b9","#c0392b","#7f8c8d","#b7950b"]
curves=[s["charts"][0]["rows"] for s in result["studies"]]
values=[float(r[k]) for rows in curves for r in rows for k in ("actual","fitted")]
low,high=min(values),max(values)
span=max(high-low,1e-10)
def points(rows,key):
    return " ".join(f"{40+700*i/max(len(rows)-1,1):.2f},{300-250*(float(r[key])-low)/span:.2f}" for i,r in enumerate(rows))
parts=['<svg xmlns="http://www.w3.org/2000/svg" width="1000" height="350" viewBox="0 0 1000 350"><rect width="1000" height="350" fill="white"/>']
parts.append(f'<polyline points="{points(curves[0],"actual")}" fill="none" stroke="#111" stroke-width="2"/>')
for study,rows,color in zip(result["studies"],curves,colors):
    parts.append(f'<polyline points="{points(rows,"fitted")}" fill="none" stroke="{color}" stroke-width="1.5"/>')
    parts.append(f'<text x="755" y="{30+22*colors.index(color)}" font-size="10" fill="{color}">{escape(study["selected"])}</text>')
parts.append('<text x="40" y="330" font-size="12">Orden mensual; precios centrados en USD/EUR. Negro: datos observados.</text></svg>')
display(SVG("".join(parts)))


## 4. Guarda el diseño y los resultados

El JSON del diseño se puede importar en la aplicación. El CSV conserva el benchmark y los cierres mensuales usados en esta ejecución; no guarda pesos preentrenados.


In [ ]:
from datetime import datetime
output = root/"reports/local/notebook-eurusd"/(preset+"-"+("synthetic" if offline else "yfinance")+"-"+datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
output.mkdir(parents=True,exist_ok=True)
(output/"blueprint.json").write_text(json.dumps({"format":"quantlabs-blueprint-library","schemaVersion":1,"entries":[{**entry,"graph":graph}]},ensure_ascii=False,indent=2))
benchmark.to_csv(output/"benchmark.csv",index=False)
runner.outputs["monthly"].to_csv(output/"monthly-inputs.csv")
print(output)
